# Tổng hợp kết quả trích xuất và đánh giá luật

Notebook này đọc trực tiếp artifact trong `outputs_holdout/k4`, `outputs_holdout/k5` và hiệu năng đã download tại `outputs_LV/rules_extraction_results`. Không tạo lại các bảng hiệu năng k4/k5.

In [ ]:
from pathlib import Path
import json
import math
import re
import pandas as pd

try:
    ROOT = Path.cwd()
    if not (ROOT / 'outputs_holdout').exists():
        ROOT = ROOT.parent
except NameError:
    ROOT = Path('.')

HOLDOUT = ROOT / 'outputs_holdout'
PERF = ROOT / 'outputs_LV' / 'rules_extraction_results'
EXPORT = ROOT / 'outputs_LV' / 'rule_extraction_summary'
EXPORT.mkdir(parents=True, exist_ok=True)

def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def mean_col(df, col):
    return float(pd.to_numeric(df[col], errors='coerce').mean()) if col in df and not df.empty else math.nan

# Existing W&B exports are the authoritative source for model performance.
performance = []
for k in (4, 5):
    p = PERF / f'k{k}.csv'
    if not p.exists():
        raise FileNotFoundError(p)
    df = pd.read_csv(p)
    df.insert(0, 'k', k)
    performance.append(df)
performance = pd.concat(performance, ignore_index=True)

count_rows, quality_rows, ruleset_rows = [], [], []
comparison_rows, gene_rows, shap_rows, case_rows, manifest = [], [], [], [], []

for k in (4, 5):
    kroot = HOLDOUT / f'k{k}'
    for dsroot in sorted(p for p in kroot.iterdir() if p.is_dir() and p.name != 'feature_selection'):
        dataset = dsroot.name
        split_info = read_json(dsroot / 'split_info.json') if (dsroot / 'split_info.json').exists() else {}
        methods = []
        for fs in ('boruta', 'mrmr_k50', 'mrmr_k75'):
            model_data = {}
            for model in ('dt', 'rf'):
                rdir = dsroot / fs / model / 'rules'
                sfile = rdir / 'rules_summary.json'
                if not sfile.exists():
                    continue
                summary = read_json(sfile)
                train = pd.read_csv(rdir / 'rules.csv') if (rdir / 'rules.csv').exists() else pd.DataFrame()
                test = pd.read_csv(rdir / 'rules_test_eval.csv') if (rdir / 'rules_test_eval.csv').exists() else pd.DataFrame()
                nraw = summary.get('n_rules_raw', math.nan)
                nkept = summary.get('n_rules_kept', summary.get('n_rules', math.nan))
                count_rows.append({
                    'dataset': dataset, 'k': k, 'fs_method': fs, 'model': model,
                    'n_rules_raw': nraw, 'n_rules_passed_filter': summary.get('n_rules_passed_filter', math.nan),
                    'n_rules_final': nkept, 'n_dropped_by_filter': summary.get('n_dropped_by_filter', math.nan),
                    'n_dropped_by_cap': summary.get('n_dropped_by_cap', math.nan),
                    'reduction_percent': 100 * (nraw - nkept) / nraw if nraw else math.nan,
                })
                quality_rows.append({
                    'dataset': dataset, 'k': k, 'fs_method': fs, 'model': model,
                    'n_rules_final': len(train), 'mean_support_train': mean_col(train, 'support'),
                    'mean_confidence_train': mean_col(train, 'confidence'), 'mean_lift_train': mean_col(train, 'lift'),
                    'mean_fidelity_train': mean_col(train, 'fidelity'), 'mean_conditions': mean_col(train, 'n_conditions'),
                    'mean_support_test': mean_col(test, 'support_test'), 'mean_confidence_test': mean_col(test, 'confidence_test'),
                    'mean_lift_test': mean_col(test, 'lift_test'), 'mean_fidelity_test': mean_col(test, 'fidelity_test'),
                    'test_metric_note': 'Per-rule mean; not set-level coverage',
                })
                ruleset_rows.append({
                    'dataset': dataset, 'k': k, 'fs_method': fs, 'model': model,
                    'n_rules_final': summary.get('n_rules_kept', summary.get('n_rules', math.nan)),
                    'coverage_train': summary.get('coverage', math.nan),
                    'mean_fidelity_train': summary.get('mean_fidelity', math.nan),
                    'n_train': split_info.get('n_train', math.nan), 'n_test': split_info.get('n_test', math.nan),
                    'coverage_test': math.nan,
                    'coverage_test_note': 'Not exported as a set-level metric; rules_test_eval contains per-rule values',
                })
                model_data[model] = (rdir, summary, train)
                gfile = rdir / 'genes_in_rules.csv'
                if gfile.exists():
                    genes = pd.read_csv(gfile)
                    for _, row in genes.iterrows():
                        gene_rows.append({'dataset': dataset, 'k': k, 'fs_method': fs, 'model': model, **row.to_dict()})
                shfile = rdir / 'shap' / 'text' / 'shap_feature_importance_matrix.csv'
                if shfile.exists() and gfile.exists():
                    sh = pd.read_csv(shfile)
                    gene_set = set(genes['gene'].dropna().astype(str))
                    if 'Feature_Gene' in sh.columns:
                        top = set(sh.sort_values('Overall_Mean_Abs', ascending=False).head(20)['Feature_Gene'].astype(str))
                        inter = gene_set & top
                        union = gene_set | top
                        shap_rows.append({'dataset': dataset, 'k': k, 'fs_method': fs, 'model': model,
                            'rule_gene_count': len(gene_set), 'shap_top_k': 20, 'shap_gene_count': len(top),
                            'common_gene_count': len(inter), 'jaccard': len(inter) / len(union) if union else 0.0,
                            'common_genes': ';'.join(sorted(inter))})
            if len(model_data) == 2:
                for compare_file in (dsroot / fs / 'compare' / 'crosscheck.json',):
                    if compare_file.exists():
                        c = read_json(compare_file)
                        comparison_rows.append({'dataset': dataset, 'k': k, 'fs_method': fs, **c})
        # Candidate cases: strongest short rules, with train/test metrics retained.
        for fs in ('boruta', 'mrmr_k50', 'mrmr_k75'):
            for model in ('dt', 'rf'):
                rdir = dsroot / fs / model / 'rules'
                a, b = rdir / 'rules.csv', rdir / 'rules_test_eval.csv'
                if not (a.exists() and b.exists()): continue
                train, test = pd.read_csv(a), pd.read_csv(b)
                if train.empty: continue
                x = train.copy(); y = test.copy()
                x['rank_score'] = x['confidence'].fillna(0) * x['fidelity'].fillna(0) * x['support'].fillna(0)
                for idx in x.sort_values(['rank_score', 'n_conditions'], ascending=[False, True]).head(3).index:
                    pos = train.index.get_loc(idx); tr = train.loc[idx]; te = test.iloc[pos]
                    case_rows.append({'dataset': dataset, 'k': k, 'fs_method': fs, 'model': model,
                        'antecedents': tr.get('antecedents'), 'consequents': tr.get('consequents'),
                        'n_conditions': tr.get('n_conditions'), 'support_train': tr.get('support'),
                        'confidence_train': tr.get('confidence'), 'fidelity_train': tr.get('fidelity'),
                        'support_test': te.get('support_test'), 'confidence_test': te.get('confidence_test'),
                        'fidelity_test': te.get('fidelity_test'), 'selection_score': tr.get('rank_score')})
        manifest.append({'k': k, 'dataset': dataset, 'split_info': str(dsroot / 'split_info.json'), 'source_root': str(dsroot)})

def save(rows, name):
    pd.DataFrame(rows).to_csv(EXPORT / name, index=False, encoding='utf-8-sig')

save(count_rows, 'rule_count_before_after.csv')
save(quality_rows, 'rule_quality_train_test.csv')
save(ruleset_rows, 'ruleset_coverage_fidelity.csv')
save(comparison_rows, 'dt_rf_rule_comparison.csv')
save(gene_rows, 'rule_gene_frequency.csv')
save(shap_rows, 'rule_shap_overlap.csv')
save(case_rows, 'case_study_candidates.csv')
save(manifest, 'manifest.csv')
print('Performance source: outputs_LV/rules_extraction_results/k4.csv, k5.csv')
print('Exported:', EXPORT)
for p in sorted(EXPORT.glob('*.csv')):
    print(p.name, pd.read_csv(p).shape)

# Optional figures used by Section 4. Performance figures are not recreated here.
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    CHARTS = EXPORT / 'charts'
    CHARTS.mkdir(exist_ok=True)
    q = pd.DataFrame(quality_rows)
    c = pd.DataFrame(count_rows)
    for kval in (4, 5):
        d = q[q.k == kval].copy()
        piv = d.pivot_table(index='dataset', columns='model', values='mean_fidelity_test', aggfunc='mean')
        ax = piv.plot(kind='bar', figsize=(14, 6), rot=65, title=f'Rule-model test fidelity (k={kval})')
        ax.set_ylabel('Mean per-rule fidelity on test')
        plt.tight_layout(); plt.savefig(CHARTS / f'rule_model_test_fidelity_k{kval}.png', dpi=220); plt.close()
    piv = c.pivot_table(index=['dataset', 'k', 'fs_method'], columns='model', values='n_rules_final', aggfunc='mean')
    ax = piv.plot(kind='bar', figsize=(16, 7), rot=70, title='Final rule count: Decision Tree versus Random Forest')
    ax.set_ylabel('Final rule count'); plt.tight_layout(); plt.savefig(CHARTS / 'dt_rf_final_rule_count.png', dpi=220); plt.close()
    s = pd.DataFrame(shap_rows)
    if not s.empty:
        piv = s.pivot_table(index='dataset', columns='k', values='jaccard', aggfunc='mean')
        ax = piv.plot(kind='bar', figsize=(14, 6), rot=65, title='Rule genes versus SHAP top-20 genes')
        ax.set_ylabel('Jaccard'); plt.tight_layout(); plt.savefig(CHARTS / 'rule_shap_jaccard.png', dpi=220); plt.close()
except Exception as exc:
    print(f'Charts were skipped ({type(exc).__name__}); CSV exports were still created.')